# Bayesian Linear Tuning

This notebook tunes the first NumPyro/JAX Bayesian linear baseline without changing the slope-core feature family. It focuses on likelihood and variational-guide choices: fixed Student-t tails, Laplace residuals, heteroskedastic scale, and AutoNormal versus AutoDelta MAP-style inference.

## 1. Setup

In [1]:
"Configure imports, paths, JAX, and deterministic settings."

from pathlib import Path
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import jax
import jax.numpy as jnp
import numpyro
import numpyro.distributions as dist
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoDelta, AutoNormal
from numpyro.optim import Adam

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = next(parent for parent in Path.cwd().parents if (parent / "src").exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.validation import group_well_folds
from src.io.loaders import discover_wells, load_horizontal_well
from src.utils.run_bundle import RunBundle

warnings.filterwarnings("ignore", category=RuntimeWarning)

RUN_TAG = "bayesian_linear_tuning"
RUN_DATE = "2026-06-22"
SEED = 42
N_SPLITS = 3
MAX_TARGET_ROWS_PER_WELL = 140
MAX_TRAIN_ROWS_PER_FOLD = 60000
SVI_STEPS = 1100
LEARNING_RATE = 0.02
POSTERIOR_SAMPLES = 140
RESIDUAL_SAMPLE_PER_FOLD = 1400

DATA_DIR = ROOT / "data" / "raw"
TRAIN_DIR = DATA_DIR / "train"
ARTIFACT_DIR = ROOT / "artifacts"
TABLE_DIR = ARTIFACT_DIR / "tables"
FIGURE_DIR = ARTIFACT_DIR / "figures"
REPORT_DIR = ARTIFACT_DIR / "reports"

for directory in [TABLE_DIR, FIGURE_DIR, REPORT_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

np.random.seed(SEED)
numpyro.set_platform("cpu")
plt.style.use("seaborn-v0_8-whitegrid")
print({"jax": jax.__version__, "numpyro": numpyro.__version__, "devices": [str(device) for device in jax.devices()]})

C:\Users\sardo\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


{'jax': '0.10.2', 'numpyro': '0.21.0', 'devices': ['cpu:0']}


## 2. Slope-Core Feature Frame

In [2]:
"Define notebook-local slope-core feature construction helpers."


def safe_numeric(series):
    return pd.to_numeric(series, errors="coerce").astype(float)


def deterministic_sample_positions(mask, max_rows):
    positions = np.flatnonzero(mask.to_numpy())
    if len(positions) <= max_rows:
        return positions
    return positions[np.linspace(0, len(positions) - 1, max_rows).astype(int)]


def deterministic_row_cap(frame, max_rows):
    if len(frame) <= max_rows:
        return frame
    positions = np.linspace(0, len(frame) - 1, max_rows).astype(int)
    return frame.iloc[positions].copy()


def window_slope(md, values, end_pos, window):
    start_pos = max(0, end_pos - window + 1)
    x0 = float(md.iloc[start_pos])
    x1 = float(md.iloc[end_pos])
    y0 = float(values.iloc[start_pos])
    y1 = float(values.iloc[end_pos])
    if not np.isfinite([x0, x1, y0, y1]).all() or abs(x1 - x0) < 1e-9:
        return np.nan
    return (y1 - y0) / (x1 - x0)


def build_slope_core_features_for_well(well, directory, max_rows):
    horizontal = load_horizontal_well(directory / f"{well}__horizontal_well.csv")
    target_mask = horizontal["TVT_input"].isna() & horizontal["TVT"].notna()
    known_mask = horizontal["TVT_input"].notna() & horizontal["TVT"].notna()
    if int(target_mask.sum()) == 0 or int(known_mask.sum()) == 0:
        return pd.DataFrame()

    tvt_input = safe_numeric(horizontal["TVT_input"])
    md = safe_numeric(horizontal["MD"])
    x = safe_numeric(horizontal["X"])
    y = safe_numeric(horizontal["Y"])
    z = safe_numeric(horizontal["Z"])

    known_positions = np.flatnonzero(known_mask.to_numpy())
    last_known_pos = int(known_positions[-1])
    last_known_tvt = float(tvt_input.iloc[last_known_pos])
    last_known_md = float(md.iloc[last_known_pos])
    last_known_z = float(z.iloc[last_known_pos])

    slope_windows = [10, 25, 50, 100]
    tvt_slopes = {f"tvt_slope_last_{window}": window_slope(md, tvt_input, last_known_pos, window) for window in slope_windows}
    z_slopes = {f"z_slope_last_{window}": window_slope(md, z, last_known_pos, window) for window in slope_windows}
    horizontal_step = np.sqrt(x.diff().fillna(0.0) ** 2 + y.diff().fillna(0.0) ** 2)
    horizontal_dist = horizontal_step.cumsum()

    target_positions = deterministic_sample_positions(target_mask, max_rows)
    all_target_positions = np.flatnonzero(target_mask.to_numpy())
    eval_rank = {int(pos): rank for rank, pos in enumerate(all_target_positions)}
    eval_denominator = max(len(all_target_positions) - 1, 1)

    rows = []
    for pos in target_positions:
        distance_md = float(md.iloc[pos] - last_known_md)
        row = {
            "wellname": well,
            "row_idx": int(horizontal.index[pos]),
            "md_norm": float((md.iloc[pos] - md.min()) / (md.max() - md.min() + 1e-9)),
            "distance_from_boundary_md": distance_md,
            "eval_progress": float(eval_rank[int(pos)] / eval_denominator),
            "last_known_tvt": last_known_tvt,
            "dZ_from_boundary": float(z.iloc[pos] - last_known_z),
            "horizontal_dist_from_boundary": float(horizontal_dist.iloc[pos] - horizontal_dist.iloc[last_known_pos]),
            "TVT": float(horizontal["TVT"].iloc[pos]),
            "endpoint_hold_pred": last_known_tvt,
        }
        row["delta_tvt_from_boundary"] = row["TVT"] - last_known_tvt
        for name, value in tvt_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
            row[f"{name}_projected_delta"] = row[name] * distance_md if np.isfinite(row[name]) else np.nan
        for name, value in z_slopes.items():
            row[name] = float(value) if np.isfinite(value) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)


slope_features = ["tvt_slope_last_10", "tvt_slope_last_25", "tvt_slope_last_50", "tvt_slope_last_100", "z_slope_last_25", "z_slope_last_50", "z_slope_last_100"]
slope_projection_features = ["tvt_slope_last_10_projected_delta", "tvt_slope_last_25_projected_delta", "tvt_slope_last_50_projected_delta", "tvt_slope_last_100_projected_delta"]
feature_columns = ["distance_from_boundary_md", "eval_progress", "md_norm", "dZ_from_boundary", "horizontal_dist_from_boundary"] + slope_features + slope_projection_features + ["last_known_tvt"]
sigma_feature_columns = ["distance_from_boundary_md", "eval_progress", "horizontal_dist_from_boundary", "md_norm"]

print({"n_mean_features": len(feature_columns), "n_sigma_features": len(sigma_feature_columns)})

{'n_mean_features': 17, 'n_sigma_features': 4}


In [3]:
"Build the sampled target-zone modeling frame."

start = time.time()
train_wells = discover_wells(TRAIN_DIR)
frames = [build_slope_core_features_for_well(well, TRAIN_DIR, MAX_TARGET_ROWS_PER_WELL) for well in train_wells]
feature_frame = pd.concat([frame for frame in frames if not frame.empty], ignore_index=True)
build_seconds = time.time() - start

print({"wells": len(train_wells), "sample_rows": len(feature_frame), "build_seconds": round(build_seconds, 2)})
feature_frame.head()

12:27:20 | INFO     | src.io.loaders | Discovered 773 wells in C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\data\raw\train


{'wells': 773, 'sample_rows': 108220, 'build_seconds': 62.85}


,wellname,row_idx,md_norm,distance_from_boundary_md,eval_progress,last_known_tvt,dZ_from_boundary,horizontal_dist_from_boundary,TVT,endpoint_hold_pred,...,tvt_slope_last_25,tvt_slope_last_25_projected_delta,tvt_slope_last_50,tvt_slope_last_50_projected_delta,tvt_slope_last_100,tvt_slope_last_100_projected_delta,z_slope_last_10,z_slope_last_25,z_slope_last_50,z_slope_last_100
0,000d7d20,1442,0.273261,1.0,0.000000,11747.37,0.01,1.000450,11747.38,11747.37,...,0.013333,0.013333,0.016327,0.016327,0.008485,0.008485,0.007778,0.005833,0.002653,0.004343
1,000d7d20,1469,0.278378,28.0,0.007040,11747.37,0.39,27.992356,11747.52,11747.37,...,0.013333,0.373333,0.016327,0.457143,0.008485,0.237576,0.007778,0.005833,0.002653,0.004343
2,000d7d20,1497,0.283684,56.0,0.014342,11747.37,1.02,55.983960,11747.46,11747.37,...,0.013333,0.746667,0.016327,0.914286,0.008485,0.475152,0.007778,0.005833,0.002653,0.004343
3,000d7d20,1524,0.288800,83.0,0.021382,11747.37,1.80,82.975967,11748.05,11747.37,...,0.013333,1.106667,0.016327,1.355102,0.008485,0.704242,0.007778,0.005833,0.002653,0.004343
4,000d7d20,1552,0.294106,111.0,0.028683,11747.37,2.68,110.962778,11748.66,11747.37,...,0.013333,1.480000,0.016327,1.812245,0.008485,0.941818,0.007778,0.005833,0.002653,0.004343


## 3. Tuned Bayesian Model Definitions

In [4]:
"Define tuned NumPyro likelihood variants and fold helpers."


def linear_mu(X):
    n_features = X.shape[1]
    intercept = numpyro.sample("intercept", dist.Normal(0.0, 1.5))
    beta = numpyro.sample("beta", dist.Normal(0.0, 2.0).expand([n_features]).to_event(1))
    return intercept + jnp.dot(X, beta)


def sigma_constant():
    return numpyro.sample("sigma", dist.HalfNormal(1.5))


def sigma_heteroskedastic(X_sigma):
    n_sigma_features = X_sigma.shape[1]
    sigma_intercept = numpyro.sample("sigma_intercept", dist.Normal(-0.1, 1.0))
    gamma = numpyro.sample("gamma", dist.Normal(0.0, 0.5).expand([n_sigma_features]).to_event(1))
    return jax.nn.softplus(sigma_intercept + jnp.dot(X_sigma, gamma)) + 1e-4


def student_t_fixed3_linear(X, X_sigma, y=None):
    mu = linear_mu(X)
    sigma = sigma_constant()
    numpyro.deterministic("mu", mu)
    numpyro.sample("obs", dist.StudentT(3.0, mu, sigma), obs=y)


def student_t_fixed5_linear(X, X_sigma, y=None):
    mu = linear_mu(X)
    sigma = sigma_constant()
    numpyro.deterministic("mu", mu)
    numpyro.sample("obs", dist.StudentT(5.0, mu, sigma), obs=y)


def student_t_fixed3_hetero(X, X_sigma, y=None):
    mu = linear_mu(X)
    sigma = sigma_heteroskedastic(X_sigma)
    numpyro.deterministic("mu", mu)
    numpyro.deterministic("sigma_row", sigma)
    numpyro.sample("obs", dist.StudentT(3.0, mu, sigma), obs=y)


def student_t_learned_hetero(X, X_sigma, y=None):
    mu = linear_mu(X)
    sigma = sigma_heteroskedastic(X_sigma)
    nu = numpyro.sample("nu", dist.Exponential(1.0 / 8.0)) + 2.0
    numpyro.deterministic("mu", mu)
    numpyro.deterministic("sigma_row", sigma)
    numpyro.deterministic("nu_effective", nu)
    numpyro.sample("obs", dist.StudentT(nu, mu, sigma), obs=y)


def laplace_linear(X, X_sigma, y=None):
    mu = linear_mu(X)
    scale = numpyro.sample("scale", dist.HalfNormal(1.5))
    numpyro.deterministic("mu", mu)
    numpyro.sample("obs", dist.Laplace(mu, scale), obs=y)


candidate_specs = [
    {"candidate": "student_t_fixed3_autonormal", "model": student_t_fixed3_linear, "guide": "autonormal", "likelihood": "student_t_fixed3", "heteroskedastic": False},
    {"candidate": "student_t_fixed5_autonormal", "model": student_t_fixed5_linear, "guide": "autonormal", "likelihood": "student_t_fixed5", "heteroskedastic": False},
    {"candidate": "student_t_fixed3_hetero_autonormal", "model": student_t_fixed3_hetero, "guide": "autonormal", "likelihood": "student_t_fixed3", "heteroskedastic": True},
    {"candidate": "student_t_learned_hetero_autonormal", "model": student_t_learned_hetero, "guide": "autonormal", "likelihood": "student_t_learned", "heteroskedastic": True},
    {"candidate": "laplace_linear_autonormal", "model": laplace_linear, "guide": "autonormal", "likelihood": "laplace", "heteroskedastic": False},
    {"candidate": "student_t_fixed3_autodelta", "model": student_t_fixed3_linear, "guide": "autodelta", "likelihood": "student_t_fixed3", "heteroskedastic": False},
]


def make_guide(spec):
    if spec["guide"] == "autodelta":
        return AutoDelta(spec["model"])
    return AutoNormal(spec["model"])


def fit_preprocessor(train_frame, columns):
    medians = train_frame[columns].median(numeric_only=True)
    filled = train_frame[columns].fillna(medians)
    means = filled.mean()
    stds = filled.std().replace(0.0, 1.0).fillna(1.0)
    return medians, means, stds


def apply_preprocessor(frame, columns, medians, means, stds):
    filled = frame[columns].fillna(medians)
    return ((filled - means) / stds).to_numpy(float)


def prepare_fold_arrays(train_frame, valid_frame):
    x_medians, x_means, x_stds = fit_preprocessor(train_frame, feature_columns)
    s_medians, s_means, s_stds = fit_preprocessor(train_frame, sigma_feature_columns)
    X_train = apply_preprocessor(train_frame, feature_columns, x_medians, x_means, x_stds)
    X_valid = apply_preprocessor(valid_frame, feature_columns, x_medians, x_means, x_stds)
    Xs_train = apply_preprocessor(train_frame, sigma_feature_columns, s_medians, s_means, s_stds)
    Xs_valid = apply_preprocessor(valid_frame, sigma_feature_columns, s_medians, s_means, s_stds)
    y_train_raw = train_frame["delta_tvt_from_boundary"].to_numpy(float)
    y_mean = float(np.mean(y_train_raw))
    y_std = float(np.std(y_train_raw) + 1e-9)
    y_train = (y_train_raw - y_mean) / y_std
    return {"X_train": jnp.asarray(X_train), "X_valid": jnp.asarray(X_valid), "Xs_train": jnp.asarray(Xs_train), "Xs_valid": jnp.asarray(Xs_valid), "y_train": jnp.asarray(y_train), "y_mean": y_mean, "y_std": y_std}


def metric_row(y_true, pred_tvt):
    return {"rmse": float(np.sqrt(mean_squared_error(y_true, pred_tvt))), "mae": float(mean_absolute_error(y_true, pred_tvt)), "r2": float(r2_score(y_true, pred_tvt))}


def interval_metrics(y_true, draws):
    q10 = np.quantile(draws, 0.10, axis=0)
    q90 = np.quantile(draws, 0.90, axis=0)
    q025 = np.quantile(draws, 0.025, axis=0)
    q975 = np.quantile(draws, 0.975, axis=0)
    return {"coverage_80": float(((y_true >= q10) & (y_true <= q90)).mean()), "coverage_95": float(((y_true >= q025) & (y_true <= q975)).mean()), "width_80": float(np.mean(q90 - q10)), "width_95": float(np.mean(q975 - q025))}

pd.DataFrame([{k: spec[k] for k in ["candidate", "guide", "likelihood", "heteroskedastic"]} for spec in candidate_specs])

,candidate,guide,likelihood,heteroskedastic
0,student_t_fixed3_autonormal,autonormal,student_t_fixed3,False
1,student_t_fixed5_autonormal,autonormal,student_t_fixed5,False
2,student_t_fixed3_hetero_autonormal,autonormal,student_t_fixed3,True
3,student_t_learned_hetero_autonormal,autonormal,student_t_learned,True
4,laplace_linear_autonormal,autonormal,laplace,False
5,student_t_fixed3_autodelta,autodelta,student_t_fixed3,False


## 4. Grouped Bayesian Tuning CV

In [5]:
"Run SVI grouped well-fold validation for tuned Bayesian candidates."

fit_start = time.time()
fold_rows = []
residual_rows = []
posterior_rows = []
wells = sorted(feature_frame["wellname"].unique())
folds = group_well_folds(wells, n_splits=N_SPLITS)

for fold_idx, (train_fold_wells, valid_fold_wells) in enumerate(folds, start=1):
    train_frame = feature_frame.loc[feature_frame["wellname"].isin(train_fold_wells)].copy()
    valid_frame = feature_frame.loc[feature_frame["wellname"].isin(valid_fold_wells)].copy()
    train_frame = deterministic_row_cap(train_frame, MAX_TRAIN_ROWS_PER_FOLD)
    y_true_tvt = valid_frame["TVT"].to_numpy(float)
    endpoint_pred = valid_frame["endpoint_hold_pred"].to_numpy(float)
    fold_rows.append({"fold": fold_idx, "candidate": "endpoint_hold", "guide": "baseline", "likelihood": "baseline", "heteroskedastic": False, "n_train_rows": 0, "n_valid_rows": len(valid_frame), "final_loss": np.nan, "coverage_80": np.nan, "coverage_95": np.nan, "width_80": np.nan, "width_95": np.nan, **metric_row(y_true_tvt, endpoint_pred)})
    arrays = prepare_fold_arrays(train_frame, valid_frame)

    for spec_idx, spec in enumerate(candidate_specs):
        rng_key = jax.random.PRNGKey(SEED + fold_idx * 100 + spec_idx)
        guide = make_guide(spec)
        svi = SVI(spec["model"], guide, Adam(LEARNING_RATE), Trace_ELBO())
        svi_result = svi.run(rng_key, SVI_STEPS, arrays["X_train"], arrays["Xs_train"], arrays["y_train"], progress_bar=False)
        params = svi_result.params
        final_loss = float(svi_result.losses[-1])
        pred_key = jax.random.PRNGKey(SEED + fold_idx * 1000 + spec_idx)
        predictive = Predictive(spec["model"], guide=guide, params=params, num_samples=POSTERIOR_SAMPLES, return_sites=("mu", "obs", "sigma_row", "nu_effective"))
        pred = predictive(pred_key, arrays["X_valid"], arrays["Xs_valid"], None)
        mu_draws_delta = np.asarray(pred["mu"]) * arrays["y_std"] + arrays["y_mean"]
        obs_draws_delta = np.asarray(pred["obs"]) * arrays["y_std"] + arrays["y_mean"]
        last_known = valid_frame["last_known_tvt"].to_numpy(float)[None, :]
        tvt_mu_draws = mu_draws_delta + last_known
        tvt_obs_draws = obs_draws_delta + last_known
        pred_tvt = tvt_mu_draws.mean(axis=0)
        metrics = metric_row(y_true_tvt, pred_tvt)
        intervals = interval_metrics(y_true_tvt, tvt_obs_draws)
        fold_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "guide": spec["guide"], "likelihood": spec["likelihood"], "heteroskedastic": spec["heteroskedastic"], "n_train_rows": len(train_frame), "n_valid_rows": len(valid_frame), "final_loss": final_loss, **metrics, **intervals})
        posterior_rows.append({"fold": fold_idx, "candidate": spec["candidate"], "final_loss": final_loss, "mean_sigma_row": float(np.asarray(pred.get("sigma_row", np.array([[np.nan]]))).mean()), "mean_nu": float(np.asarray(pred.get("nu_effective", np.array([np.nan]))).mean())})
        sample_idx = np.linspace(0, len(valid_frame) - 1, min(len(valid_frame), RESIDUAL_SAMPLE_PER_FOLD)).astype(int)
        residual_rows.append(pd.DataFrame({"fold": fold_idx, "candidate": spec["candidate"], "wellname": valid_frame.iloc[sample_idx]["wellname"].to_numpy(), "row_idx": valid_frame.iloc[sample_idx]["row_idx"].to_numpy(), "distance_from_boundary_md": valid_frame.iloc[sample_idx]["distance_from_boundary_md"].to_numpy(float), "eval_progress": valid_frame.iloc[sample_idx]["eval_progress"].to_numpy(float), "y_true": y_true_tvt[sample_idx], "y_pred": pred_tvt[sample_idx], "residual": y_true_tvt[sample_idx] - pred_tvt[sample_idx], "posterior_sd": tvt_obs_draws[:, sample_idx].std(axis=0)}))

fit_seconds = time.time() - fit_start
fold_metrics = pd.DataFrame(fold_rows)
summary = (fold_metrics.groupby(["candidate", "guide", "likelihood", "heteroskedastic"], as_index=False).agg(rmse_mean=("rmse", "mean"), rmse_std=("rmse", "std"), mae_mean=("mae", "mean"), r2_mean=("r2", "mean"), coverage_80_mean=("coverage_80", "mean"), coverage_95_mean=("coverage_95", "mean"), width_80_mean=("width_80", "mean"), width_95_mean=("width_95", "mean"), n_valid_rows=("n_valid_rows", "sum"), n_train_rows_mean=("n_train_rows", "mean"), final_loss_mean=("final_loss", "mean")).sort_values("rmse_mean"))
residual_sample = pd.concat(residual_rows, ignore_index=True)
posterior_diagnostics = pd.DataFrame(posterior_rows)

print({"fit_seconds": round(fit_seconds, 2), "fold_rows": len(fold_metrics), "residual_rows": len(residual_sample)})
summary

{'fit_seconds': 209.51, 'fold_rows': 21, 'residual_rows': 25200}


,candidate,guide,likelihood,heteroskedastic,rmse_mean,rmse_std,mae_mean,r2_mean,coverage_80_mean,coverage_95_mean,width_80_mean,width_95_mean,n_valid_rows,n_train_rows_mean,final_loss_mean
2,student_t_fixed3_autodelta,autodelta,student_t_fixed3,False,14.844948,0.520500,10.392001,0.999425,0.769440,0.935504,31.122894,59.155521,108220,60000.0,77370.747396
3,student_t_fixed3_autonormal,autonormal,student_t_fixed3,False,14.931658,0.498154,10.469386,0.999418,0.773957,0.937497,31.665274,60.164355,108220,60000.0,78261.651042
1,laplace_linear_autonormal,autonormal,laplace,False,14.934259,0.519391,10.449149,0.999418,0.783881,0.937769,32.712354,59.368867,108220,60000.0,77427.875000
4,student_t_fixed3_hetero_autonormal,autonormal,student_t_fixed3,True,14.982206,0.515556,10.538824,0.999415,0.780500,0.941445,32.537254,61.842499,108220,60000.0,74781.752604
5,student_t_fixed5_autonormal,autonormal,student_t_fixed5,False,15.013122,0.460230,10.527899,0.999412,0.774434,0.921681,31.862986,54.359453,108220,60000.0,78372.916667
6,student_t_learned_hetero_autonormal,autonormal,student_t_learned,True,15.139998,0.587346,10.648057,0.999402,0.777243,0.939197,32.800576,62.361422,108220,60000.0,74879.026042
0,endpoint_hold,baseline,baseline,False,15.578175,0.651555,10.905266,0.999367,NaN,NaN,NaN,NaN,108220,0.0,NaN


## 5. Artifacts And Diagnostics

In [6]:
"Export tuning metrics, diagnostics, and figures."

fold_metrics_path = TABLE_DIR / f"model__bayesian_linear_tuning_fold_metrics__{RUN_DATE}.csv"
summary_path = TABLE_DIR / f"model__bayesian_linear_tuning_summary__{RUN_DATE}.csv"
posterior_path = TABLE_DIR / f"model__bayesian_linear_tuning_posterior_diagnostics__{RUN_DATE}.csv"
residual_path = TABLE_DIR / f"model__bayesian_linear_tuning_residual_sample__{RUN_DATE}.csv"
fold_metrics.to_csv(fold_metrics_path, index=False)
summary.to_csv(summary_path, index=False)
posterior_diagnostics.to_csv(posterior_path, index=False)
residual_sample.to_csv(residual_path, index=False)

fig_paths = {}
plot_summary = summary.sort_values("rmse_mean", ascending=True)
fig, ax = plt.subplots(figsize=(10, 6))
colors = ["#4C78A8" if candidate != "endpoint_hold" else "#F58518" for candidate in plot_summary["candidate"]]
ax.barh(plot_summary["candidate"], plot_summary["rmse_mean"], color=colors)
ax.set_title("Bayesian Linear Tuning RMSE")
ax.set_xlabel("grouped target-zone RMSE")
ax.set_ylabel("")
fig.tight_layout()
fig_paths["rmse_comparison"] = FIGURE_DIR / f"model__bayesian_linear_tuning_rmse_comparison__{RUN_DATE}.png"
fig.savefig(fig_paths["rmse_comparison"], dpi=160)
plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
for candidate, group in fold_metrics.groupby("candidate"):
    if candidate == "endpoint_hold" or candidate in set(summary.head(4)["candidate"]):
        ax.plot(group["fold"], group["rmse"], marker="o", label=candidate)
ax.set_title("Bayesian Linear Tuning Fold RMSE")
ax.set_xlabel("fold")
ax.set_ylabel("RMSE")
ax.legend(fontsize=7)
fig.tight_layout()
fig_paths["fold_rmse"] = FIGURE_DIR / f"model__bayesian_linear_tuning_fold_rmse__{RUN_DATE}.png"
fig.savefig(fig_paths["fold_rmse"], dpi=160)
plt.close(fig)

bayes_only = summary.loc[summary["candidate"] != "endpoint_hold"].copy()
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(bayes_only["width_80_mean"], bayes_only["coverage_80_mean"], s=80, color="#54A24B")
for row in bayes_only.itertuples(index=False):
    ax.text(row.width_80_mean, row.coverage_80_mean, row.candidate, fontsize=7)
ax.axhline(0.80, color="black", linewidth=1, linestyle="--")
ax.set_title("80% Coverage vs Width")
ax.set_xlabel("mean 80% interval width")
ax.set_ylabel("empirical 80% coverage")
fig.tight_layout()
fig_paths["coverage_width"] = FIGURE_DIR / f"model__bayesian_linear_tuning_coverage_width__{RUN_DATE}.png"
fig.savefig(fig_paths["coverage_width"], dpi=160)
plt.close(fig)

best_candidate = str(bayes_only.iloc[0]["candidate"])
best_residuals = residual_sample.loc[residual_sample["candidate"] == best_candidate]
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(best_residuals["distance_from_boundary_md"], best_residuals["posterior_sd"], s=7, alpha=0.35, color="#B279A2")
ax.set_title(f"Posterior SD vs Boundary Distance: {best_candidate}")
ax.set_xlabel("MD distance past boundary")
ax.set_ylabel("posterior predictive SD")
fig.tight_layout()
fig_paths["uncertainty_by_boundary"] = FIGURE_DIR / f"model__bayesian_linear_tuning_uncertainty_by_boundary__{RUN_DATE}.png"
fig.savefig(fig_paths["uncertainty_by_boundary"], dpi=160)
plt.close(fig)

{key: str(value) for key, value in fig_paths.items()}

{'rmse_comparison': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_linear_tuning_rmse_comparison__2026-06-22.png',
 'fold_rmse': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_linear_tuning_fold_rmse__2026-06-22.png',
 'coverage_width': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_linear_tuning_coverage_width__2026-06-22.png',
 'uncertainty_by_boundary': 'C:\\Users\\sardo\\OneDrive\\Desktop\\Classes\\Projects\\Wellbore-tvt-Modeling\\artifacts\\figures\\model__bayesian_linear_tuning_uncertainty_by_boundary__2026-06-22.png'}

In [7]:
"Write report, update comparison table, and create run bundle."

bayes_summary = summary.loc[summary["candidate"] != "endpoint_hold"].copy()
best = bayes_summary.iloc[0]
endpoint = summary.loc[summary["candidate"] == "endpoint_hold"].iloc[0]
previous_bayes_rmse = 14.975233756558557
glm_huber_rmse = 14.792842528391505

findings = pd.DataFrame([
    {"topic": "best_tuned_bayesian_candidate", "finding": f"Best tuned Bayesian candidate is {best.candidate} with sampled grouped RMSE {best.rmse_mean:.4f}.", "modeling_implication": "Use this to decide whether Bayesian linear tuning meaningfully improved the first Bayesian baseline."},
    {"topic": "baseline_comparison", "finding": f"Best tuned Bayesian RMSE {best.rmse_mean:.4f} vs previous Bayesian best {previous_bayes_rmse:.4f} and Huber GLM {glm_huber_rmse:.4f}.", "modeling_implication": "If Bayesian linear still trails Huber, move to HSGP residual structure rather than more linear tuning."},
    {"topic": "coverage_check", "finding": f"Best tuned Bayesian 80% coverage is {best.coverage_80_mean:.3f} with mean width {best.width_80_mean:.2f} TVT units.", "modeling_implication": "Coverage remains the main Bayesian value even if posterior mean RMSE is not leading."},
])
findings_path = TABLE_DIR / f"model__bayesian_linear_tuning_findings__{RUN_DATE}.csv"
findings.to_csv(findings_path, index=False)

report_path = REPORT_DIR / f"model__bayesian_linear_tuning__{RUN_DATE}.md"
report_lines = ["# Bayesian Linear Tuning", "", f"Date: {RUN_DATE}", "", "## Key Findings"]
for row in findings.itertuples(index=False):
    report_lines.extend([f"- **{row.topic}**: {row.finding}", f"  - Modeling implication: {row.modeling_implication}"])
report_lines.extend(["", "## Best Candidate", f"- Candidate: `{best.candidate}`", f"- Guide: `{best.guide}`", f"- Likelihood: `{best.likelihood}`", f"- Heteroskedastic: `{bool(best.heteroskedastic)}`", f"- RMSE mean/std: {best.rmse_mean:.4f} / {best.rmse_std:.4f}", f"- 80% coverage/width: {best.coverage_80_mean:.3f} / {best.width_80_mean:.2f}", "", "## Artifacts", f"- Fold metrics: {fold_metrics_path}", f"- Summary: {summary_path}", f"- Posterior diagnostics: {posterior_path}", f"- Residual sample: {residual_path}", f"- Findings: {findings_path}"])
for name, fig_path in fig_paths.items():
    report_lines.append(f"- {name}: {fig_path}")
report_path.write_text("\n".join(report_lines) + "\n", encoding="utf-8")

comparison_path = TABLE_DIR / "model_comparison.csv"
comparison = pd.read_csv(comparison_path)
run_comparison_row = pd.DataFrame([{"run_id": "pending_run_bundle", "model_family": "bayesian_linear_tuning", "model_name": str(best.candidate), "cv_rmse_mean": float(best.rmse_mean), "cv_rmse_std": float(best.rmse_std), "train_time_s": float(build_seconds + fit_seconds), "n_features": int(len(feature_columns)), "notes": "NumPyro/JAX Bayesian linear tuning over likelihood tails and guides"}])
comparison = comparison.loc[comparison["run_id"] != "pending_run_bundle"]
comparison = pd.concat([comparison, run_comparison_row], ignore_index=True)
comparison.to_csv(comparison_path, index=False)

config = {"task": "bayesian_linear_tuning", "seed": SEED, "n_splits": N_SPLITS, "max_target_rows_per_well": MAX_TARGET_ROWS_PER_WELL, "max_train_rows_per_fold": MAX_TRAIN_ROWS_PER_FOLD, "svi_steps": SVI_STEPS, "learning_rate": LEARNING_RATE, "posterior_samples": POSTERIOR_SAMPLES, "best_candidate": str(best.candidate), "note": "NumPyro/JAX Bayesian linear tuning before HSGP."}
with RunBundle(RUN_TAG, config, seed=SEED) as rb:
    rb.add_input(str(TRAIN_DIR))
    rb.add_metric("sample_rows", float(len(feature_frame)))
    rb.add_metric("best_rmse", float(best.rmse_mean))
    rb.add_metric("best_rmse_std", float(best.rmse_std))
    rb.add_metric("best_coverage_80", float(best.coverage_80_mean))
    rb.add_metric("best_coverage_95", float(best.coverage_95_mean))
    rb.add_metric("endpoint_hold_rmse", float(endpoint.rmse_mean))
    rb.add_metric("best_minus_previous_bayes", float(best.rmse_mean - previous_bayes_rmse))
    rb.add_metric("best_minus_huber_glm", float(best.rmse_mean - glm_huber_rmse))
    for output_path in [fold_metrics_path, summary_path, posterior_path, residual_path, findings_path, report_path, comparison_path]:
        rb.add_output(str(output_path))
    for fig_path in fig_paths.values():
        rb.add_output(str(fig_path))
    rb.set_notes("Bayesian linear tuning completed with fixed-tail Student-t, Laplace, and AutoDelta/AutoNormal variants.")
run_id = rb.run_id
comparison.loc[comparison["run_id"] == "pending_run_bundle", "run_id"] = run_id
comparison.to_csv(comparison_path, index=False)

print({"run_id": run_id, "best_candidate": best.candidate, "best_rmse": round(float(best.rmse_mean), 4), "coverage_80": round(float(best.coverage_80_mean), 3), "best_minus_previous_bayes": round(float(best.rmse_mean - previous_bayes_rmse), 4), "best_minus_huber_glm": round(float(best.rmse_mean - glm_huber_rmse), 4)})
findings

{'run_id': '2026-06-22__1231__bayesian_linear_tuning__4b66b91', 'best_candidate': 'student_t_fixed3_autodelta', 'best_rmse': 14.8449, 'coverage_80': 0.769, 'best_minus_previous_bayes': -0.1303, 'best_minus_huber_glm': 0.0521}


,topic,finding,modeling_implication
0,best_tuned_bayesian_candidate,Best tuned Bayesian candidate is student_t_fix...,Use this to decide whether Bayesian linear tun...
1,baseline_comparison,Best tuned Bayesian RMSE 14.8449 vs previous B...,"If Bayesian linear still trails Huber, move to..."
2,coverage_check,Best tuned Bayesian 80% coverage is 0.769 with...,Coverage remains the main Bayesian value even ...


## Results

Use the executed summary above to decide whether more Bayesian linear tuning is worthwhile or whether the next Bayesian lift should come from an HSGP residual component.